# C++ Code Generator

This notebook provides a modular workflow for generating optimized C++ code from Python using a frontier LLM, with optional compile and benchmark helpers.

---

## 1. Import Required Libraries and Dependencies
Import all necessary Python packages for the project.

In [1]:
# Imports
import os
import gradio as gr
import requests
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables
load_dotenv(override=True)

load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')
BASE_URL="https://openrouter.ai/api/v1"
openai = OpenAI(api_key=api_key, base_url=BASE_URL)
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
assert api_key, "OpenRouter API key not found in environment variables!"


---

## 2. LLM Utility for C++ Generation
Define a function to call the HuggingFace Inference API for generating optimized C++ code from Python code or natural language description.

In [17]:
# LLM Utility for C++ Generation

def generate_cpp_code(input_code, optimize=True, model=MODEL_GPT):
    prompt = f"Convert the following Python code to optimized C++ code. {'Optimize for speed and memory.' if optimize else ''}\n\n{input_code}\n\nRespond with only the C++ code."
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 2048,
        "temperature": 0.3
    }
    response = openai.chat.completions.create(**payload)
    return response.model_dump_json()["message"]["content"]

---

## 3. Compile Helper Utility
Function to compile generated C++ code and capture output/errors.

In [3]:
# Compile Helper Utility
import subprocess

def compile_cpp_code(code, filename="generated.cpp"):
    with open(filename, "w") as f:
        f.write(code)
    compile_cmd = ["g++", filename, "-o", "generated_bin"]
    try:
        result = subprocess.run(compile_cmd, capture_output=True, text=True, check=True)
        return "Compilation successful", result.stdout
    except subprocess.CalledProcessError as e:
        return "Compilation failed", e.stderr

---

## 4. Benchmark Helper Utility
Function to benchmark the compiled C++ binary and report execution time.

In [4]:
# Benchmark Helper Utility
import time

def benchmark_cpp_binary(binary_path="./generated_bin"):
    start = time.time()
    try:
        result = subprocess.run([binary_path], capture_output=True, text=True, check=True)
        end = time.time()
        return f"Execution time: {end - start:.4f} seconds", result.stdout
    except subprocess.CalledProcessError as e:
        return "Execution failed", e.stderr

---

## 5. Gradio UI Construction
Build a modular Gradio interface for Python-to-C++ conversion, optimization, compile, and benchmark helpers.

In [7]:
# Gradio UI Construction

def gradio_cpp_app():
    with gr.Blocks() as demo:
        gr.Markdown("## C++ Code Generator")
        py_code = gr.Textbox(label="Enter Python code or description", lines=6)
        optimize = gr.Checkbox(label="Optimize for speed and memory", value=True)
        model = gr.Dropdown(choices=[MODEL_GPT, MODEL_LLAMA], value=MODEL_GPT, label="Select LLM Model")
        generate_btn = gr.Button("Generate C++ Code")
        cpp_code = gr.Code(label="Generated C++ Code", language="cpp")
        compile_btn = gr.Button("Compile C++ Code")
        compile_output = gr.Textbox(label="Compile Output")
        benchmark_btn = gr.Button("Benchmark Binary")
        benchmark_output = gr.Textbox(label="Benchmark Output")

        def on_generate(py, opt, model):
            return generate_cpp_code(py, opt, model=model)
        def on_compile(code):
            status, output = compile_cpp_code(code)
            return f"{status}\n{output}"
        def on_benchmark():
            status, output = benchmark_cpp_binary()
            return f"{status}\n{output}"

        generate_btn.click(on_generate, [py_code, optimize, model], cpp_code)
        compile_btn.click(on_compile, cpp_code, compile_output)
        benchmark_btn.click(on_benchmark, None, benchmark_output)
    return demo

# Example usage:
# demo = gradio_cpp_app()
# demo.launch()

---

## 6. Main Application Runner
Combine all utilities and UI components to run the application.

In [ ]:
# Main Application Runner

def main():
    demo = gradio_cpp_app()
    demo.launch()

if __name__ == '__main__':
    main()